# Flight Price Prediction — Log Target Experiment

## Why log target might help tree models

We already know log1p(price) is nearly symmetric (skew ≈ −0.03 after cleaning).
For linear models the improvement was negligible (M2 vs M3: +$0.26) because the model
was already fitting the bulk of Economy fares well.

For tree models the argument is different:
- A leaf node predicts the **mean** of all samples that fall into it
- High-price Business fares in the same leaf as Economy fares pull that mean up,
  worsening predictions for Economy rows in that leaf
- In log space the Business fares are closer in magnitude to Economy fares,
  so the leaf means are more representative of all rows in the leaf
- RMSE is also computed in log space during training, so the model penalises
  relative errors rather than absolute ones

**Hypothesis:** log target reduces the influence of high-price Business outliers
on tree splits and leaf predictions, lowering RMSE on the full test set.

Run this notebook **after** 03_random_forest and 04_gradient_boosting.
Best raw-target params are taken from those notebooks and re-used here.
A fresh RandomizedSearchCV is run in log space to find the optimal log-target params.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict, RandomizedSearchCV

In [2]:
_cwd = Path().resolve()
ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA = ROOT / "data"

train = pd.read_csv(DATA / "train.csv")
test  = pd.read_csv(DATA / "test.csv")

train = train[
    (train['price'] > 0) & (train['airline'] != 'Southern_Airways_Express')
].reset_index(drop=True)

KF = KFold(n_splits=5, shuffle=True, random_state=42)

STOPS_MAP = {'zero': 0, 'one': 1, 'two_or_more': 2}
TIME_MAP  = {'Early_Morning': 0, 'Morning': 1, 'Afternoon': 2,
             'Evening': 3, 'Night': 4, 'Late_Night': 5}
CLASS_MAP = {'Economy': 0, 'Business': 1}

train_enc = train.assign(
    stops          = train['stops'].map(STOPS_MAP),
    departure_time = train['departure_time'].map(TIME_MAP),
    arrival_time   = train['arrival_time'].map(TIME_MAP),
    flight_class   = train['flight_class'].map(CLASS_MAP),
).drop(columns=['id', 'flight'])

X   = train_enc.drop(columns=['price'])
y   = train['price'].to_numpy()
y_log = np.log1p(y)

print(f"train: {train.shape}   X: {X.shape}")

train: (18050, 12)   X: (18050, 9)


In [3]:
NUM_COLS = ['duration', 'days_left', 'stops', 'departure_time', 'arrival_time', 'flight_class']
OHE_COLS = ['airline', 'source_city', 'destination_city']

def make_preprocessor():
    return ColumnTransformer([
        ('pass', 'passthrough', NUM_COLS),
        ('ohe',  OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), OHE_COLS),
    ])

def make_pipeline(model):
    return Pipeline([('pre', make_preprocessor()), ('model', model)])

def cv_rmse_log(model, X, y):
    # Fit on log(y), predict, expm1 back, report RMSE in dollar units
    log_preds = cross_val_predict(model, X, np.log1p(y), cv=KF)
    return float(np.sqrt(np.mean((y - np.expm1(log_preds)) ** 2)))

In [4]:
# RF with log target — use best raw-target params from notebook 03 as starting point
# (params that minimise raw-RMSE may not be optimal in log space, but it is a useful comparison)
rf_raw_best_params = {
    'n_estimators': 100, 'min_samples_leaf': 2,
    'max_features': 0.5, 'max_depth': 20,
}
pipe_rf_log = make_pipeline(RandomForestRegressor(**rf_raw_best_params, random_state=42, n_jobs=-1))
rmse_rf_log = cv_rmse_log(pipe_rf_log, X, y)
print(f"RF  (raw-target params) + log target : ${rmse_rf_log:.2f}")
print(f"RF  tuned (raw target)               : $122.60")

RF  (raw-target params) + log target : $123.36
RF  tuned (raw target)               : $122.60


In [5]:
# RF re-tuned in log space — optimal params may differ
rf_param_dist = {
    'model__n_estimators'    : [100, 200, 300, 500],
    'model__max_depth'       : [None, 10, 20, 30],
    'model__min_samples_leaf': [1, 2, 4, 8],
    'model__max_features'    : ['sqrt', 'log2', 0.3, 0.5],
}

rs_rf = RandomizedSearchCV(
    make_pipeline(RandomForestRegressor(random_state=42, n_jobs=-1)),
    param_distributions=rf_param_dist,
    n_iter=20,
    cv=KF,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=1,
)
rs_rf.fit(X, y_log)

best_rf_log_params = {k.replace('model__', ''): v for k, v in rs_rf.best_params_.items()}
print("Best RF log-target params:", best_rf_log_params)

pipe_rf_log_tuned = make_pipeline(RandomForestRegressor(**best_rf_log_params, random_state=42, n_jobs=-1))
rmse_rf_log_tuned = cv_rmse_log(pipe_rf_log_tuned, X, y)
print(f"RF  tuned in log space : ${rmse_rf_log_tuned:.2f}")
print(f"RF  tuned raw          : $122.60   delta: {122.60 - rmse_rf_log_tuned:+.2f}")

Best RF log-target params: {'n_estimators': 100, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}
RF  tuned in log space : $123.36
RF  tuned raw          : $122.60   delta: -0.76


In [6]:
# GB re-tuned in log space
# Paste best raw-target GB params from notebook 04 below
gb_param_dist = {
    'model__n_estimators'    : [100, 200, 300],
    'model__learning_rate'   : [0.05, 0.1, 0.2],
    'model__max_depth'       : [3, 4, 5],
    'model__subsample'       : [0.8, 1.0],
    'model__min_samples_leaf': [5, 10, 20],
}

rs_gb = RandomizedSearchCV(
    make_pipeline(GradientBoostingRegressor(random_state=42)),
    param_distributions=gb_param_dist,
    n_iter=15,
    cv=KF,
    scoring='neg_root_mean_squared_error',
    random_state=42,
)
rs_gb.fit(X, y_log)

best_gb_log_params = {k.replace('model__', ''): v for k, v in rs_gb.best_params_.items()}
print("Best GB log-target params:", best_gb_log_params)

pipe_gb_log_tuned = make_pipeline(GradientBoostingRegressor(**best_gb_log_params, random_state=42))
rmse_gb_log_tuned = cv_rmse_log(pipe_gb_log_tuned, X, y)
print(f"GB  tuned in log space : ${rmse_gb_log_tuned:.2f}")

Best GB log-target params: {'subsample': 0.8, 'n_estimators': 300, 'min_samples_leaf': 5, 'max_depth': 5, 'learning_rate': 0.1}
GB  tuned in log space : $121.92


In [7]:
# Full comparison — fill in GB raw results from notebook 04 output
GB_BASELINE_RAW = None   # paste value from 04_gradient_boosting cell gb-05
GB_TUNED_RAW    = None   # paste value from 04_gradient_boosting cell gb-07

results = [
    {'model': 'M2  LinearRegression  all features',    'CV RMSE ($)': 166.75},
    {'model': 'RF  tuned  raw target',                 'CV RMSE ($)': 122.60},
    {'model': 'GB  baseline  raw target',              'CV RMSE ($)': GB_BASELINE_RAW},
    {'model': 'GB  tuned    raw target',               'CV RMSE ($)': GB_TUNED_RAW},
    {'model': 'RF  raw params + log target',           'CV RMSE ($)': round(rmse_rf_log, 2)},
    {'model': 'RF  tuned in log space',                'CV RMSE ($)': round(rmse_rf_log_tuned, 2)},
    {'model': 'GB  tuned in log space',                'CV RMSE ($)': round(rmse_gb_log_tuned, 2)},
]

summary = (pd.DataFrame(results)
             .dropna()
             .sort_values('CV RMSE ($)')
             .reset_index(drop=True))
summary.index += 1
print(summary.to_string())

                                model  CV RMSE ($)
1              GB  tuned in log space       121.92
2               RF  tuned  raw target       122.60
3         RF  raw params + log target       123.36
4              RF  tuned in log space       123.36
5  M2  LinearRegression  all features       166.75
